# CITE-seq: Treg Characterization and TCR Repertoire Analysis

1. **Quality filtering**: keep cells where RNA_annotation == Protein_annotation
2. **T/NK cell extraction**: cells with TCR data (RNA-based annotation == 'tnk')
3. **RNA normalization** → export raw counts for Seurat label transfer (`01_cite_seq_annotation.R`)
4. **AUCell Treg subset**: Harmony integration by dataset, UMAP, gene module scoring
5. **Pseudotime bin assignment**: Early (B0) / Late (B1–B4) from label transfer
6. **TCR repertoire analysis**: clonotype diversity, V-gene usage, clone sharing, pairing rates

**Inputs**:
- `RNA_raw_counts.csv`                            – cell × gene raw counts
- `protein_raw_counts.csv`                        – cell × protein raw counts
- `annotation_table.csv`                          – per-cell annotations + dataset
- `TCR_merged.csv`                                – merged TCR contig table
- `tcell/pr_rna_tcr_tcell_aucell.csv`            – AUCell T cell subtype labels
- `tcell/notstrict_treg_pt_transfer_labels.csv`  – pseudotime bin transfer (from `01_cite_seq_annotation.R`)

**Outputs**:
- `tcell/good_anno_t_wtcr_norm_counts.csv`  – normalized T cell counts (input to `01_cite_seq_annotation.R`)
- `tcell/notstrict_treg_raw_counts.csv`     – Treg raw counts for PT bin transfer
- TRBV gene usage barplot (Early vs Late Tregs)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import anndata as ad
import scipy.stats as stats
from scipy.stats import gmean, chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR  = "/path/to/cite-seq/output"
TCELL_DIR = f"{DATA_DIR}/tcell"

## 1. Load data and quality filter

In [ ]:
rna_counts = pd.read_csv(f"{DATA_DIR}/RNA_raw_counts.csv")
anno       = pd.read_csv(f"{DATA_DIR}/annotation_table.csv")

# Keep cells where RNA and protein annotations agree
good_cell = [
    cell for cell, rna_ann, prot_ann
    in zip(anno['Unnamed: 0'], anno['RNA_annotation'], anno['Protein_annotation'])
    if rna_ann == prot_ann
]
print(f"Quality-filtered cells: {len(good_cell):,} / {len(anno):,}")

good_anno = anno[anno['Unnamed: 0'].isin(good_cell)]
good_rna  = rna_counts[rna_counts['Unnamed: 0'].isin(good_cell)]

## 2. Extract T/NK cells with TCR and normalize RNA

In [ ]:
# Load TCR data and restrict to quality-filtered T/NK cells
tcr = pd.read_csv(f"{DATA_DIR}/TCR_merged.csv")
tcr_good = tcr[tcr['barcode'].isin(good_cell)]
tcr_good = tcr_good.merge(good_anno, left_on='barcode', right_on='Unnamed: 0', how='left')
tcr_good = tcr_good[tcr_good['Protein_annotation'] == 'tnk']

# RNA counts for T/NK cells
t_counts = good_rna[good_rna['Unnamed: 0'].isin(tcr_good['barcode'])]
t_counts.index = t_counts['Unnamed: 0'].tolist()
t_counts = t_counts.drop('Unnamed: 0', axis=1)

# Build AnnData and normalize
t_adata = ad.AnnData(t_counts.copy())
t_adata.layers['counts'] = t_counts.values
sc.pp.normalize_total(t_adata)
sc.pp.log1p(t_adata)

# Export normalized counts for AUCell T cell annotation in 01_cite_seq_annotation.R
t_norm = pd.DataFrame(t_adata.X, index=t_adata.obs_names, columns=t_adata.var_names)
t_norm.to_csv(f"{TCELL_DIR}/good_anno_t_wtcr_norm_counts.csv")
print("Exported T cell normalized counts.")

## 3. Subset AUCell-annotated Tregs and attach dataset metadata

In [ ]:
# Load AUCell T cell labels (from 01_cite_seq_annotation.R)
aucell_anno = pd.read_csv(f"{TCELL_DIR}/pr_rna_tcr_tcell_aucell.csv")
t_adata.obs['anno'] = aucell_anno['label'].tolist()

# Subset Tregs
treg = t_adata[t_adata.obs['anno'] == 'Treg'].copy()

# Attach dataset metadata for Harmony batch correction
ds = good_anno[good_anno['Unnamed: 0'].isin(treg.obs.index)][['Unnamed: 0', 'dataset']]
treg.obs = treg.obs.merge(ds, how='left', left_index=True, right_on='Unnamed: 0')

print(treg)

## 4. Gene module scoring

In [ ]:
panels = {
    'Activated_tissue': ['BATF', 'PRDM1', 'MAF', 'TNFRSF18', 'TNFRSF4'],
    'Tfr':              ['BCL6', 'CXCR5', 'PDCD1', 'ICOS'],
    'Th17_like':        ['RORC', 'CCR6', 'IL17A', 'IL17F'],
    'IFN_response':     ['ISG15', 'IFI6', 'MX1', 'IFIT3'],
    'Cell_cycle':       ['MKI67', 'TOP2A', 'STMN1'],
    'Treg_core':        ['FOXP3', 'IL2RA', 'CTLA4', 'TIGIT', 'ENTPD1', 'LRRC32', 'IL10']
}
for name, genes in panels.items():
    sc.tl.score_genes(treg, [g for g in genes if g in treg.var_names], score_name=name)

## 5. Harmony integration and UMAP

In [ ]:
# HVG selection on raw counts
treg_hvg = treg.copy()
sc.pp.filter_genes(treg_hvg, min_cells=5)
sc.pp.highly_variable_genes(treg_hvg, n_top_genes=3000, flavor='cell_ranger', layer='counts')
treg.var['highly_variable'] = False
treg.var.loc[treg_hvg.var_names, 'highly_variable'] = treg_hvg.var['highly_variable'].values

sc.pp.scale(treg, max_value=10)

# Harmony integration by dataset
sce.pp.harmony_integrate(treg, key='dataset')
sc.pp.neighbors(treg, use_rep='X_pca_harmony', n_neighbors=20, n_pcs=30)
sc.tl.umap(treg)

# Visualize gene modules
for name in panels.keys():
    sc.pl.umap(treg, color=name, cmap='viridis', vmin='p1', vmax='p99', frameon=False)

## 6. Export Treg raw counts for pseudotime bin transfer

In [ ]:
# Export raw counts → input for Seurat label transfer in 01_cite_seq_annotation.R
raw_counts = pd.DataFrame(treg.layers['counts'], index=treg.obs.index, columns=treg.var_names)
raw_counts.to_csv(f"{TCELL_DIR}/notstrict_treg_raw_counts.csv")
print("Exported Treg raw counts for pseudotime bin transfer.")

## 7. Assign pseudotime bins (Early / Late) from label transfer

In [ ]:
# Load predicted pseudotime bins from 01_cite_seq_annotation.R
treg_bin = pd.read_csv(f"{TCELL_DIR}/notstrict_treg_pt_transfer_labels.csv")
print(treg_bin['predicted.id'].value_counts())

# B0 = Early, B1-B4 = Late (matching CD Treg pseudotime bins)
treg_bin['bin'] = ['Early' if i == 0 else 'Late' for i in treg_bin['predicted.id']]

treg.obs['pt_bin']   = treg_bin['bin'].tolist()
treg.obs['pt_group'] = treg_bin['predicted.id'].tolist()

sc.pl.umap(treg, color='pt_bin',   frameon=False, palette={'Early': '#4393c3', 'Late': '#d6604d'})
sc.pl.umap(treg, color='pt_group', frameon=False, palette='tab20')

## 8. TCR repertoire analysis

In [ ]:
# Merge pt_bin into TCR data
treg.obs.index = treg.obs['Unnamed: 0'].tolist()
tcr_treg = tcr_good[tcr_good['barcode'].isin(treg.obs.index)].copy()
tcr_treg = tcr_treg.merge(treg.obs[['pt_bin']], how='left', left_on='barcode', right_index=True)

# Select relevant columns
tcr_treg = tcr_treg[[
    'barcode', 'is_cell', 'contig_id', 'high_confidence',
    'length', 'chain', 'v_gene', 'd_gene', 'j_gene', 'c_gene',
    'full_length', 'productive', 'fwr1', 'fwr1_nt', 'cdr1', 'cdr1_nt',
    'fwr2', 'fwr2_nt', 'cdr2', 'cdr2_nt', 'fwr3', 'fwr3_nt', 'cdr3',
    'cdr3_nt', 'fwr4', 'fwr4_nt', 'reads', 'umis', 'raw_clonotype_id',
    'raw_consensus_id', 'exact_subclonotype_id', 'dataset', 'pt_bin'
]]
print(tcr_treg['pt_bin'].value_counts())

In [ ]:
def pick_best_contig(x):
    """Select best contig per chain: productive first, then highest UMIs, then reads."""
    x = x.copy()
    x['productive_bool'] = x['productive'] == True
    return x.sort_values(['productive_bool', 'umis', 'reads'], ascending=[False, False, False]).iloc[0]

def summarize_cell(x):
    """Per-cell TCR summary: chain pairing, V/J gene, CDR3 sequences."""
    has_TRA = ((x['productive'] == True) & (x['chain'] == 'TRA')).any()
    has_TRB = ((x['productive'] == True) & (x['chain'] == 'TRB')).any()

    tra_best = pick_best_contig(x[x['chain'] == 'TRA']) if (x['chain'] == 'TRA').any() else None
    trb_best = pick_best_contig(x[x['chain'] == 'TRB']) if (x['chain'] == 'TRB').any() else None

    def first_nonnull(s):
        s = s.dropna()
        return s.iloc[0] if len(s) > 0 else np.nan

    out = {
        'has_productive': (x['productive'] == True).any(),
        'has_TRA': has_TRA, 'has_TRB': has_TRB,
        'has_pairing': has_TRA & has_TRB,
        'raw_clonotype_id': first_nonnull(x['raw_clonotype_id']),
        'reads_total': x['reads'].fillna(0).sum(),
        'umis_total':  x['umis'].fillna(0).sum(),
        'TRB_v_gene': trb_best['v_gene']  if trb_best is not None else np.nan,
        'TRB_j_gene': trb_best['j_gene']  if trb_best is not None else np.nan,
        'TRB_cdr3':   trb_best['cdr3']    if trb_best is not None else np.nan,
        'TRA_v_gene': tra_best['v_gene']  if tra_best is not None else np.nan,
        'TRA_cdr3':   tra_best['cdr3']    if tra_best is not None else np.nan,
    }
    return pd.Series(out)

df = tcr_treg[tcr_treg['high_confidence'] == True].copy()

cell_df = (
    df.groupby(['barcode', 'pt_bin'], sort=False)
      .apply(summarize_cell)
      .reset_index()
)

prod_cells = cell_df[cell_df['has_productive'] == True].copy()
prod_cells['clonotype'] = prod_cells['raw_clonotype_id']

In [ ]:
# Export per-cell TCR summary (Supplementary Table 9)
supp9_cols = [
    'barcode', 'pt_bin', 'clonotype',
    'has_TRA', 'has_TRB', 'has_pairing',
    'TRA_v_gene', 'TRA_cdr3',
    'TRB_v_gene', 'TRB_j_gene', 'TRB_cdr3',
    'reads_total', 'umis_total'
]
prod_cells[supp9_cols].to_csv(f"{TCELL_DIR}/treg_tcr_per_cell_summary.csv", index=False)
print(f"Exported TCR per-cell summary: {prod_cells.shape[0]:,} productive cells")

## 9. Clonotype diversity and repertoire statistics

In [ ]:
def shannon(x):
    counts = x.value_counts()
    p = counts / counts.sum()
    return -(p * np.log(p)).sum()

# Shannon entropy by pseudotime bin
diversity = prod_cells.groupby('pt_bin')['clonotype'].apply(shannon)
print("Shannon diversity (observed):")
print(diversity)

# Downsampling Early to match Late cell count for fair comparison
late_n = prod_cells[prod_cells['pt_bin'] == 'Late'].shape[0]
early  = prod_cells[prod_cells['pt_bin'] == 'Early']

np.random.seed(42)
diversities = [shannon(early.sample(late_n, replace=False)['clonotype']) for _ in range(1000)]
print(f"\nDownsampled Early diversity: {np.mean(diversities):.3f} ± {np.std(diversities):.3f}")

In [ ]:
# Top clone fraction
top_frac = prod_cells.groupby('pt_bin')['clonotype'].apply(
    lambda x: x.value_counts().max() / x.value_counts().sum()
)
print("Top clone fraction:", top_frac.to_dict())

# Clone sharing: fraction of Late clones also present in Early
late_clones  = set(prod_cells.loc[prod_cells.pt_bin == 'Late',  'clonotype'])
early_clones = set(prod_cells.loc[prod_cells.pt_bin == 'Early', 'clonotype'])
shared = len(late_clones & early_clones) / len(late_clones)
print(f"\nFraction of Late clones seen in Early: {shared:.3f}")

# Fraction of Late cells whose clone is present in Early
prod_cells['clone_seen_in_early'] = prod_cells['clonotype'].isin(early_clones)
late_cells = prod_cells[prod_cells.pt_bin == 'Late']
n_shared   = late_cells['clone_seen_in_early'].sum()
print(f"Late cells with clone in Early: {n_shared}/{len(late_cells)} ({n_shared/len(late_cells):.3f})")

In [ ]:
# Pairing rates (TRA + TRB both productive)
pairing_summary = cell_df.groupby('pt_bin')['has_pairing'].agg(['sum', 'count'])
pairing_summary['percent'] = pairing_summary['sum'] / pairing_summary['count'] * 100
print("Pairing rates (%):", pairing_summary)

# CDR3 length distribution
prod_cells['TRB_cdr3_len'] = prod_cells['TRB_cdr3'].astype(str).str.len()
print("\nTRB CDR3 length by bin:")
print(prod_cells.groupby('pt_bin')['TRB_cdr3_len'].describe())

## 10. TRBV gene usage

In [ ]:
# TRBV frequency table (column-normalized)
trbv_tab = pd.crosstab(prod_cells['TRB_v_gene'], prod_cells['pt_bin'], normalize='columns').head(15)

# Chi-squared test for TRBV usage differences
trbv_counts = pd.crosstab(prod_cells['TRB_v_gene'], prod_cells['pt_bin'])
chi2, p_chi2, dof, _ = chi2_contingency(trbv_counts)
print(f"Chi-squared TRBV usage: χ²={chi2:.2f}, p={p_chi2:.4f}, df={dof}")

In [ ]:
# Barplot: top 10 TRBV genes by frequency
top_genes = trbv_tab.sum(axis=1).sort_values(ascending=False).head(10).index

trbv_long = (
    trbv_tab
    .reset_index()
    .melt(id_vars='TRB_v_gene', var_name='pt_bin', value_name='frequency')
)
trbv_long['percentage'] = trbv_long['frequency'] * 100
trbv_top = trbv_long[trbv_long['TRB_v_gene'].isin(top_genes)]

plt.figure(figsize=(6, 4))
sns.barplot(
    data=trbv_top, x='TRB_v_gene', y='percentage', hue='pt_bin',
    palette=['#dfc6c7', '#885e7c']
)
plt.xticks(rotation=90)
plt.ylabel('Percentage of cells')
plt.xlabel('TRBV gene')
plt.title('TRBV gene usage in Early vs Late Tregs')
plt.tight_layout()
plt.show()